# T10: Tools

A **tool** is a function the LLM can ask to call. The LLM does not run it: it returns a structured request, the application runs the function and sends the result back.

This track builds up to a RAG agent step by step:

| Notebook | What it adds |
|---|---|
| **T10** | Define a tool; run the request-execute-respond loop by hand |
| T11 | `ToolAgent` runs the loop; several tools; structured output |
| T12 | Retrieval as a tool; loop by hand |
| T13 | Retrieval tool inside a `ToolAgent` |
| T14 | RAG agent wrapped as a tool of a parent agent (subagent) |

## Define a tool

A tool has a **name**, a **description** that tells the LLM when to use it, a **JSON Schema** of its parameters, and an async `call` method with the logic. `SumTwoNumbers` from `ki_kompass_rag/tool_agents.py`:

```python
class SumTwoNumbers(Tool):
    name = "sum_two_numbers"
    description = "Sum two numbers."
    parameters = _schema(
        number_1=("number", "The first number."),
        number_2=("number", "The second number."),
    )

    async def call(self, args: dict[str, Any]) -> dict[str, Any]:
        a, b = args.get("number_1"), args.get("number_2")
        if not isinstance(a, int | float) or not isinstance(b, int | float):
            raise ValueError("number_1 and number_2 must be numbers.")
        return {"result": a + b}
```

`json_schema()` is what the LLM sees. If `call` raises, `ToolAgent` (T11) sends the error message back to the LLM, which can retry with corrected arguments.

In [ ]:
import json
import sys

from loguru import logger

from conversational_toolkit.llms.base import LLMMessage, MessageContent, Roles
from conversational_toolkit.llms.openai import OpenAILLM

from ki_kompass_rag.tool_agents import SumTwoNumbers

logger.remove()
logger.add(sys.stderr, level="WARNING")

tool_sum = SumTwoNumbers()
print(json.dumps(tool_sum.json_schema(), indent=2))
print(await tool_sum.call({"number_1": 5, "number_2": 10}))

## The tool-call loop

1. **Request**: send the user message; the LLM answers with a tool call instead of text.
2. **Execute**: run the requested tools.
3. **Respond**: send the results back; the LLM writes the final answer.

With `tool_choice="auto"` the LLM may also answer directly without a tool.

### 1. Request

In [ ]:
user_message = LLMMessage(
    role=Roles.USER,
    content=[MessageContent(type="text", text="Sum the numbers 15 and 27 for me.")],
)
llm = OpenAILLM(tools=[tool_sum], tool_choice="auto")

response = await llm.generate([user_message])

print(f"Content: {response.content[0].text!r}")
for tool_call in response.tool_calls or []:
    print(
        f"Tool call {tool_call.id}: {tool_call.function.name}({tool_call.function.arguments})"
    )

### 2. Execute

The application parses the arguments and runs the tool. Results are keyed by the tool call id, because the LLM can request the same tool several times in one response.

In [ ]:
tools = {tool.name: tool for tool in llm.tools or []}
results = {}
for tool_call in response.tool_calls or []:
    args = json.loads(tool_call.function.arguments)
    results[tool_call.id] = await tools[tool_call.function.name].call(args)

print(results)

### 3. Respond

Each result goes back as a `TOOL` message linked by `tool_call_id`, after the assistant message that requested it.

In [ ]:
tool_messages = [
    LLMMessage(
        role=Roles.TOOL,
        name=tool_call.function.name,
        tool_call_id=tool_call.id,
        content=[MessageContent(type="text", text=json.dumps(results[tool_call.id]))],
    )
    for tool_call in response.tool_calls or []
]

final_response = await llm.generate([user_message, response, *tool_messages])
print(final_response.content[0].text)